In [ ]:
%load_ext autoreload
%autoreload 2
import pickle
import numpy as np
import pandas as pd
import wsb_utils as wu
import model_utils as mu

/workspaces/veritasai_project
['.git', '.gitignore', 'README.md', '__pycache__', 'model_utils.py', 'prices.pkl', 'project.ipynb', 'requirements.txt', 'wsb_utils.py']


In [ ]:
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=43373, as_frame=True, parser="auto").frame
print(df.shape)
df.columns

In [ ]:
url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt"
sym = pd.read_csv(url, sep="|")
valid_tickers = set(sym["Symbol"].dropna().astype(str))
print(len(valid_tickers), "symbols")

(53187, 7)


Index(['title', 'score', 'url', 'comms_num', 'created', 'body', 'timestamp'],
      dtype='str')

In [ ]:
start = wu.first_full_day(df, date_col="created")
print("Data starts properly on:", start.date())

days = wu.to_trading_day(df["created"])
df_full = df[days.values >= start]
print(len(df_full), "posts kept of", len(df))

mentions = wu.daily_mentions(df_full, valid_tickers, date_col="created")
spikes = wu.find_spikes(mentions, ratio=5, min_mentions=5)
print(len(spikes), "spikes")
spikes.head(20)

13285 symbols


In [ ]:
post_days = wu.to_trading_day(df_full["created"])
print("Posts: ", post_days.min().date(), "→", post_days.max().date())
print("Spikes:", spikes["date"].min().date(), "→", spikes["date"].max().date())

print("\nPosts per month:")
print(post_days.dt.to_period("M").value_counts().sort_index())

print("\nTop 40 most-mentioned symbols:")
print(mentions.groupby("ticker")["mentions"].sum().sort_values(ascending=False).head(40))

print("\nSpikes per stock:")
print(spikes["ticker"].value_counts().head(30))

print("\nSpikes per month:")
print(spikes["date"].dt.to_period("M").value_counts().sort_index())

Data starts properly on: 2021-01-28
53186 posts kept of 53187
122 spikes


,date,ticker,mentions,avg_before
0,2021-02-11,ACB,5,0.857143
1,2021-02-11,TLRY,21,2.500000
2,2021-02-12,TSM,6,0.500000
3,2021-02-15,IBM,5,0.500000
4,2021-02-15,INTC,7,0.428571
5,2021-02-15,IP,5,0.428571
6,2021-02-15,MVIS,6,0.428571
7,2021-02-15,NVDA,7,0.571429
8,2021-02-18,RIOT,5,0.571429
9,2021-02-19,PLTR,37,7.142857


In [ ]:
for s in ["ASO", "GM", "IP", "AG"]:
    titles = df_full[df_full["title"].str.contains(rf"\b{s}\b", na=False)]["title"].head(5).tolist()
    print(s, titles, "\n")

Posts:  2021-01-28 → 2021-08-16
Spikes: 2021-02-11 → 2021-08-16

Posts per month:
created
2021-01    16829
2021-02    19404
2021-03     7740
2021-04     1989
2021-05     2261
2021-06     2748
2021-07     1364
2021-08      851
Freq: M, Name: count, dtype: int64

Top 40 most-mentioned symbols:
ticker
GME     12055
AMC      4485
BB       1866
NOK      1364
PLTR      752
RKT       715
TSLA      551
UWMC      382
AMD       348
SNDL      338
CLOV      308
SPCE      274
AAPL      213
NIO       204
SLV       200
TLRY      175
CLNE      169
CRSR      166
CLF       164
AAL       156
MVIS      149
AMZN      147
WKHS      144
GM        143
ASO       129
BABA      118
DKNG      107
MSFT       98
NVDA       97
FB         96
SOFI       89
IBKR       84
JPM        83
AG         82
MUST       78
GE         76
KOSS       75
WEEK       70
ARKK       70
CASH       69
Name: mentions, dtype: int64

Spikes per stock:
ticker
PLTR    9
RKT     9
UWMC    8
MVIS    6
BB      6
AMD     6
ASO     5
AMC     5
AAPL 

In [ ]:
results = []
for ratio in [3, 4, 5]:
    for min_m in [5, 8, 10]:
        n = len(wu.find_spikes(mentions, ratio=ratio, min_mentions=min_m))
        results.append({"ratio": ratio, "min_mentions": min_m, "spikes": n})
pd.DataFrame(results).pivot(index="ratio", columns="min_mentions", values="spikes")

ASO ['ASO STOCKS', 'ASO is free money.', '(ASO) Academy Sports and Outdoors Next RKT similar fundamentals', 'ASO - DD - First timer - Spreading some knowledge', "Half my account into ASO calls today! Scared money don't make money! "] 

GM ['BUY GM', 'What is we made a thread about GM.E not GME.', 'Stock Market News for Today  GM / SQ / UBER  The Short Squeeze  other Stock Market News [01-29]', 'BREAKING: Robinhood now limiting trading in 50+ stocks, including GM, Beyond Meat, and Starbucks', 'Ford Motor Company, will they respond to GM with an all electric model lineup?'] 

IP ['Complete DD on HYLN. No battery patent, no IP, no product. Nikola 2.0?', 'Yolo on Pure options 1TDE. If GME IS 500 on 2-26 I will put the WSB logo on ice cream cups packaging I mass manufacture (can someone give me some advice on the IP?)', 'BUY LABU/XBI FAAAAAAAST - MERKEL OPPOSES IP WAVER', 'Potential waiving of IP rights for Covid-19 Vaccines'] 

AG ["Once you're done with GME - AG and SLV, the gentleman's s

In [ ]:
price_start = spikes["date"].min() - pd.Timedelta(days=60)   # history for the clues
price_end   = spikes["date"].max() + pd.Timedelta(days=15)   # future for the label

tickers = sorted(spikes["ticker"].unique()) + ["SPY"]
prices = mu.get_prices(tickers, price_start, price_end)

print(len(prices), "of", len(tickers), "tickers downloaded")
print("Missing:", sorted(set(tickers) - set(prices)))

pickle.dump(prices, open("prices.pkl", "wb"))

min_mentions,5,8,10
ratio,,,
3,184,103,78
4,145,77,58
5,122,64,46


In [ ]:
data = mu.build_dataset(spikes, prices)
print(len(data), "spikes with enough price history")

data = data.sort_values(["ticker", "date"])
gap = data.groupby("ticker")["date"].diff().dt.days
data = data[gap.isna() | (gap > 7)].dropna(subset=mu.FEATURES)
data = data.sort_values("date").reset_index(drop=True)
print(len(data), "examples after removing repeats")

49 of 50 tickers downloaded
Missing: ['HOOD']


In [ ]:
for t in [-0.10, -0.15, -0.20]:
    hit = data["min_future_ret"] <= t
    print(f"fall of {abs(t):.0%}+: {hit.sum()} drops ({hit.mean():.0%} of spikes)")

DROP = -0.10
data["crashed"] = (data["min_future_ret"] <= DROP).astype(int)
print("\nSharp-drop rate:", round(data["crashed"].mean(), 3), "| drops:", data["crashed"].sum())

117 spikes with enough price history
81 examples after removing repeats


In [ ]:
cutoff = data["date"].iloc[int(len(data) * 0.7)]

train = data[data["date"] < cutoff - pd.Timedelta(days=7)]
test  = data[data["date"] >= cutoff]

print("Cutoff:", cutoff.date())
print(f"Train: {len(train)} spikes, {train['crashed'].sum()} drops  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Test:  {len(test)} spikes, {test['crashed'].sum()} drops  ({test['date'].min().date()} → {test['date'].max().date()})")

X_train, y_train = train[mu.FEATURES], train["crashed"]
X_test,  y_test  = test[mu.FEATURES],  test["crashed"]

fall of 10%+: 28 drops (35% of spikes)
fall of 15%+: 14 drops (17% of spikes)
fall of 20%+: 7 drops (9% of spikes)

Sharp-drop rate: 0.346 | drops: 28


In [ ]:
from sklearn.metrics import precision_score, recall_score, average_precision_score

def report(name, y, pred):
    pred = np.asarray(pred).astype(int)
    print(f"{name:28s} precision {precision_score(y, pred, zero_division=0):.2f}   "
          f"recall {recall_score(y, pred, zero_division=0):.2f}   warnings {pred.sum()}/{len(y)}")

report("Always warn",            y_test, np.ones(len(y_test)))
report("Rule: volume 3x normal", y_test, test["volume_ratio"] >= 3)
report("Rule: already up 50%+",  y_test, test["ret_5d"] >= 0.5)

Cutoff: 2021-06-11
Train: 47 spikes, 18 drops  (2021-02-11 → 2021-05-28)
Test:  28 spikes, 9 drops  (2021-06-11 → 2021-08-16)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                             class_weight="balanced", random_state=42),
}

probas = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probas[name] = model.predict_proba(X_test)[:, 1]
    report(name, y_test, probas[name] >= 0.5)
    print(f"{'':28s} PR-AUC {average_precision_score(y_test, probas[name]):.2f}  "
          f"(random guessing ≈ {y_test.mean():.2f})")

Always warn                  precision 0.32   recall 1.00   warnings 28/28
Rule: volume 3x normal       precision 0.71   recall 0.56   warnings 7/28
Rule: already up 50%+        precision 1.00   recall 0.22   warnings 2/28


In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(),
                             LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                             class_weight="balanced", random_state=42),
}

probas = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probas[name] = model.predict_proba(X_test)[:, 1]
    report(name, y_test, probas[name] >= 0.5)
    print(f"{'':28s} PR-AUC {average_precision_score(y_test, probas[name]):.2f}  "
          f"(random guessing ≈ {y_test.mean():.2f})")

Logistic regression          precision 0.57   recall 0.44   warnings 7/28
                             PR-AUC 0.65  (random guessing ≈ 0.32)
Random forest                precision 0.80   recall 0.89   warnings 10/28
                             PR-AUC 0.92  (random guessing ≈ 0.32)
